In [1]:
%pip install -qU langchain langchain-core langchain-groq gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 570.0/570.0 kB 28.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.8/30.8 MB 30.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 8.3 MB/s eta 0:00:00


In [2]:
import os

try:
    from google.colab import userdata
    groq_api_key = userdata.get("GROQ_API_KEY") or userdata.get("GROQ_API_KEY")
except Exception:
    groq_api_key = None

if groq_api_key:
    os.environ["GROQ_API_KEY"] = groq_api_key
    print("Groq API key configured successfully.")
else:
    raise ValueError("GROQ_API_KEY not found in Colab Secrets.")

Groq API key configured successfully.


In [3]:
from langchain_groq import ChatGroq

model = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0,
    max_tokens=300
)

In [4]:
from langchain_core.prompts import PromptTemplate

research_template = PromptTemplate(
    template="""
Please summarize the research paper titled "{paper_input}" with the following specifications:
Explanation Style: {style_input}
Explanation Length: {length_input}

Requirements:
1. Explain the main idea clearly.
2. Mention the problem the paper is trying to solve.
3. Mention the method or model used in the paper.
4. Mention important mathematical ideas only if they are relevant.
5. Use simple analogies when helpful.
6. If certain information is not available, respond with: "Insufficient information available" instead of guessing.

Ensure the summary is clear, accurate, and aligned with the selected style and length.
""",
    input_variables=["paper_input", "style_input", "length_input"],
    validate_template=True,
)

print("Research prompt template created successfully.")


Research prompt template created successfully.


In [8]:
import gradio as gr

research_chain = research_template | model

papers = [
    "Attention Is All You Need",
    "BERT: Pre-training of Deep Bidirectional Transformers",
    "GPT-3: Language Models are Few-Shot Learners",
    "Diffusion Models Beat GANs on Image Synthesis",

]

styles = ["Beginner-Friendly", "Technical", "Code-Oriented", "Mathematical"]
lengths = ["Short (1-2 paragraphs)", "Medium (3-5 paragraphs)", "Long (detailed explanation)"]


def summarize_paper(paper_input, style_input, length_input):
    result = research_chain.invoke({
        "paper_input": paper_input,
        "style_input": style_input,
        "length_input": length_input,
    })
    return result.content

research_demo = gr.Interface(
    fn=summarize_paper,
    inputs=[
        gr.Dropdown(papers, label="Select Research Paper Name"),
        gr.Dropdown(styles, label="Select Explanation Style"),
        gr.Dropdown(lengths, label="Select Explanation Length"),
    ],
    outputs=gr.Markdown(label="Summary"),
    title="Research Paper Summarizer using Groq + LangChain",
    description="A Colab-friendly Gradio version of the original Streamlit research tool.",
)

research_demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://3488d67c306a8565a7.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://3488d67c306a8565a7.gradio.live
